In [ ]:
# -*- coding: utf-8 -*-
!pip install -q wfdb scipy 'numpy<2.0' 'pandas==2.2.2' tensorflow seaborn matplotlib scikit-learn

import os, json, csv, random, shutil, time, tarfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
from scipy import signal as sp_signal
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)

print(f'TensorFlow : {tf.__version__}')
print(f'wfdb       : {wfdb.__version__}')
print('All imports OK.')

from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

# ECG Arrhythmia Classifier — MIT-BIH + PTB-XL + LTAFDB + NSTDB

All ECG data is streamed from PhysioNet.

### Pipeline overview
| Block | Step |
|-------|------|
| 1 | Setup — installs, imports, paths, constants |
| 2 | Extract NSR / AFIB from MIT-BIH |
| 3 | Extract PVC / LBBB from MIT-BIH |
| 3b | Extract NSR / AFIB / LBBB from PTB-XL (500 Hz → 360 Hz) |
| 3c | Extract AFIB / PVC from Long-Term AF Database (128 Hz → 360 Hz) |
| 3d | Extract PVC / LBBB from INCART 12-lead DB (257 Hz → 360 Hz) |
| 4 | Patient-wise train / val / test split |
| 5 | Augment training set with NSTDB noise |
| 6 | Filter & z-score normalise every segment |
| 7 | Train dual-input CNN + evaluation — early stopping on val macro-F1 |
| 8 | TFLite conversion — float32 + dynamic-range quantised |

---
## Block 1 — Setup

In [ ]:
BASE_DIR           = '/content/drive/MyDrive/ecg_data'
DATASETS_TAR_DRIVE = '/content/drive/MyDrive/DATASETS.tar.gz'
RUNS_DIR           = '/content/drive/MyDrive/ecg_runs'
SPLIT_DIR          = os.path.join(BASE_DIR, 'split')
AUG_DIR            = os.path.join(BASE_DIR, 'split-augmented')
FILTERED_DIR       = os.path.join(BASE_DIR, 'split-filtered')
SEGS_TAR_DRIVE     = os.path.join(BASE_DIR, 'segs.tar.gz')
AUG_TAR_DRIVE      = os.path.join(BASE_DIR, 'aug_segs.tar.gz')

TMP_BASE      = '/tmp/ecg'
TMP_RAW_DIR   = os.path.join(TMP_BASE, 'raw')
NSR_AFIB_DIR  = os.path.join(TMP_BASE, 'segs', 'nsr-afib')
PVC_LBBB_DIR  = os.path.join(TMP_BASE, 'segs', 'pvc-lbbb')
AUG_SEGS_DIR  = os.path.join(TMP_BASE, 'aug_segs')

MITDB_LOCAL   = os.path.join(TMP_RAW_DIR, 'mitdb')
PTBXL_LOCAL   = os.path.join(TMP_RAW_DIR, 'ptb-xl')
LTAFDB_LOCAL  = os.path.join(TMP_RAW_DIR, 'ltafdb')
INCART_LOCAL  = os.path.join(TMP_RAW_DIR, 'incartdb')

WIN_LEN        = 1024
HALF_WIN       = WIN_LEN // 2
SAMPLE_RATE_HZ = 360.0
CLASSES        = ['NSR', 'AFIB', 'PVC', 'LBBB']
CLASS_TO_IDX   = {cls: idx for idx, cls in enumerate(CLASSES)}
N_WORKERS      = 2

os.makedirs(BASE_DIR, exist_ok=True)

for _d in (
    TMP_RAW_DIR,
    NSR_AFIB_DIR,
    PVC_LBBB_DIR,
    os.path.join(NSR_AFIB_DIR, 'NSR'),
    os.path.join(NSR_AFIB_DIR, 'AFIB'),
    os.path.join(PVC_LBBB_DIR, 'PVC'),
    os.path.join(PVC_LBBB_DIR, 'LBBB'),
    AUG_SEGS_DIR,
):
    os.makedirs(_d, exist_ok=True)

In [ ]:
PROGRESS_FILE = os.path.join(BASE_DIR, 'progress.json')

def _load_progress():
    try:
        with open(PROGRESS_FILE) as _f:
            return json.load(_f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def _is_done(step):
    return _load_progress().get(step, False)

def _mark_done(step):
    _p = _load_progress()
    _p[step] = True
    _payload = json.dumps(_p, indent=2)
    for _try in range(5):
        try:
            with open(PROGRESS_FILE, 'w') as _f:
                _f.write(_payload)
                _f.flush()
                os.fsync(_f.fileno())
            print(f'  [✓] Checkpoint saved: {step!r}')
            return
        except OSError as _e:
            print(f'  [WARN] progress write failed (attempt {_try+1}/5): {_e}')
            time.sleep(10)
    print(f'  [ERROR] Could not save checkpoint for {step!r} after 5 tries.')


def _extract_tar(tar_path, dst_dir):
    """Extract a tar.gz archive to dst_dir."""
    os.makedirs(dst_dir, exist_ok=True)
    with tarfile.open(tar_path, 'r:gz') as _tf:
        _tf.extractall(dst_dir)

def _restore_from_drive(tar_drive, dst_dir):
    """Copy a tar.gz from Drive to /tmp, then extract."""
    _tmp_tar = f'/tmp/_restore_{os.path.basename(tar_drive)}'
    _sz = os.path.getsize(tar_drive) / 1e6
    print(f'  Copying {os.path.basename(tar_drive)} from Drive ({_sz:.0f} MB) ...')
    shutil.copy2(tar_drive, _tmp_tar)
    print(f'  Extracting to {dst_dir} ...')
    _extract_tar(_tmp_tar, dst_dir)
    os.remove(_tmp_tar)
    print(f'  Done.')

def _compress_to_drive(src_dir, tar_drive, arc_name=None):
    """Compress a local directory to tar.gz and write it to Drive."""
    _tmp_tar = f'/tmp/_save_{os.path.basename(tar_drive)}'
    _arc = arc_name or os.path.basename(src_dir)
    print(f'  Compressing {src_dir} ...')
    with tarfile.open(_tmp_tar, 'w:gz') as _tf:
        _tf.add(src_dir, arcname=_arc)
    _sz = os.path.getsize(_tmp_tar) / 1e6
    print(f'  Copying to Drive ({_sz:.0f} MB compressed) ...')
    shutil.copy2(_tmp_tar, tar_drive)
    os.remove(_tmp_tar)
    print(f'  Saved: {tar_drive}')

In [ ]:
_EXTRACTION_STEPS = ('block2', 'block3', 'block3b', 'block3c', 'block3d')
_extraction_done  = all(_is_done(s) for s in _EXTRACTION_STEPS)

if _extraction_done:
    _segs_ok = (
        os.path.isdir(os.path.join(NSR_AFIB_DIR, 'NSR'))
        and bool(os.listdir(os.path.join(NSR_AFIB_DIR, 'NSR')))
    )
    if not _segs_ok:
        if os.path.exists(SEGS_TAR_DRIVE):
            print('Restoring extraction segments from Drive (segs.tar.gz) ...')
            _restore_from_drive(SEGS_TAR_DRIVE, TMP_BASE)
            print('Segments restored to /tmp.')
        else:
            print('[WARN] All extraction blocks are marked done but segs.tar.gz is missing.')
            print('       Reset extraction steps and re-run to rebuild.')
    else:
        print('✓ Extraction segments already in /tmp.')
else:
    if os.path.isdir(MITDB_LOCAL) and os.listdir(MITDB_LOCAL):
        print('✓ Raw datasets already unpacked.')
    elif os.path.exists(DATASETS_TAR_DRIVE):
        _sz = os.path.getsize(DATASETS_TAR_DRIVE) / 1e6
        print(f'Unpacking DATASETS.tar.gz ({_sz:.0f} MB) ...')
        _restore_from_drive(DATASETS_TAR_DRIVE, TMP_RAW_DIR)
        print('Datasets unpacked.')
    else:
        print('[INFO] DATASETS.tar.gz not found — all blocks will stream from PhysioNet.')

if _is_done('block5'):
    _aug_ok = os.path.isdir(AUG_SEGS_DIR) and bool(os.listdir(AUG_SEGS_DIR))
    if not _aug_ok:
        if os.path.exists(AUG_TAR_DRIVE):
            print('Restoring augmented segments from Drive (aug_segs.tar.gz) ...')
            _restore_from_drive(AUG_TAR_DRIVE, TMP_BASE)
            print('Augmented segments restored to /tmp.')
        else:
            print('[WARN] block5 done but aug_segs.tar.gz missing — reset block5 to re-run.')
    else:
        print('✓ Augmented segments already in /tmp.')


def _has_hea_files(path):
    try:
        return os.path.isdir(path) and any(f.endswith('.hea') for f in os.listdir(path))
    except OSError:
        return False

def _wfdb_args(local_dir, rec_name, remote_db):
    """Return (record_arg, pn_dir_arg) for wfdb calls."""
    if _has_hea_files(local_dir):
        return os.path.join(local_dir, rec_name), None
    return rec_name, remote_db


MITDB_PN   = MITDB_LOCAL  if _has_hea_files(MITDB_LOCAL)  else 'mitdb'
LTAFDB_PN  = LTAFDB_LOCAL if _has_hea_files(LTAFDB_LOCAL) else 'ltafdb'
INCART_PN  = INCART_LOCAL if _has_hea_files(INCART_LOCAL) else 'incartdb'
_r500          = os.path.join(PTBXL_LOCAL, 'records500')
PTBXL_LOCAL_OK = bool(os.path.isdir(_r500) and os.listdir(_r500))

print('BASE_DIR      :', BASE_DIR)
print('TMP_BASE      :', TMP_BASE)
print('NSR_AFIB_DIR  :', NSR_AFIB_DIR)
print('PVC_LBBB_DIR  :', PVC_LBBB_DIR)
print('MITDB_PN      :', MITDB_PN)
print('LTAFDB_PN     :', LTAFDB_PN)
print('INCART_PN     :', INCART_PN)
print('PROGRESS_FILE :', PROGRESS_FILE)
_existing   = _load_progress()
_done_steps = [k for k, v in _existing.items() if v]
print(f'Completed steps: {_done_steps or "none"}')

In [ ]:
if _has_hea_files(MITDB_LOCAL):
    mitdb_records = sorted(
        os.path.splitext(f)[0] for f in os.listdir(MITDB_LOCAL) if f.endswith('.hea')
    )
    print(f'MIT-BIH Arrhythmia DB: {len(mitdb_records)} records (local)')
else:
    print('Fetching MIT-BIH record list from PhysioNet ...')
    mitdb_records = wfdb.get_record_list('mitdb')
    print(f'MIT-BIH Arrhythmia DB: {len(mitdb_records)} records (PhysioNet)')

print('Records:', sorted(mitdb_records))
print('\nBlock 1 complete — proceed to Block 2.')

---
## Block 2 — Extract NSR / AFIB Segments

Streams MIT-BIH records from PhysioNet (`mitdb`).

- Rhythm boundaries from `+` annotations: `(N`/`(NSR` → NSR, `(AFIB` → AFIB.
- Non-overlapping 1 024-sample windows per interval.
- NSR windows: all beats must be `N` or `.`.
- AFIB windows: rejected if any ventricular/ectopy symbol is present (`V`, `r`, `E`, `F`, `L`, `R`, `B`, `!`, `[`, `]`).
- Records without MLII channel are skipped.

Output: `nsr-afib/NSR/`, `nsr-afib/AFIB/`

In [ ]:
def get_mlii_channel(sig_name_list):
    """Return 0-based index of the MLII channel, or None if absent."""
    for i, name in enumerate(sig_name_list):
        if name.strip().upper() in ('MLII', 'ML II', 'II'):
            return i
    return None


_STEP = 'block2'
if _is_done(_STEP):
    print(f'✓ Block 2 already done — skipping.')
else:
    RHYTHM_MAP = {
        '(N':    'NSR',
        '(NSR':  'NSR',
        '(AFIB': 'AFIB',
    }
    NORMAL_BEAT_SYMBOLS = {'N', '.'}

    AFIB_EXCL_SYMBOLS = {'V', 'r', 'E', 'F', 'L', 'R', 'B', '!', '[', ']'}

    for folder in ('NSR', 'AFIB'):
        p = os.path.join(NSR_AFIB_DIR, folder)
        shutil.rmtree(p, ignore_errors=True)
        os.makedirs(p, exist_ok=True)

    total_counts = {'NSR': 0, 'AFIB': 0}

    for rec in mitdb_records:
        _r, _pn = _wfdb_args(MITDB_LOCAL, rec, 'mitdb')
        try:
            hdr = wfdb.rdheader(_r, pn_dir=_pn)
        except Exception as e:
            print(f'  [SKIP] {rec}: cannot read header — {e}')
            continue

        ch = get_mlii_channel(hdr.sig_name)
        if ch is None:
            print(f'  [SKIP] {rec}: no MLII channel')
            continue

        try:
            signal_obj = wfdb.rdrecord(_r, channels=[ch], pn_dir=_pn)
            signal = signal_obj.p_signal[:, 0].astype(np.float32)
        except Exception as e:
            print(f'  [WARN] {rec}: cannot read signal — {e}')
            continue

        try:
            ann = wfdb.rdann(_r, 'atr', pn_dir=_pn)
        except Exception as e:
            print(f'  [WARN] {rec}: cannot read annotations — {e}')
            continue

        n_samples  = len(signal)
        beat_index = list(zip(ann.sample, ann.symbol))

        rhythm_events = []
        for sample_idx, symbol, aux in zip(ann.sample, ann.symbol, ann.aux_note):
            if symbol == '+':
                label = aux.strip().rstrip('\x00').strip()
                rhythm_events.append((sample_idx, label))

        if not rhythm_events:
            print(f'  {rec}: no rhythm annotations — skipping')
            continue

        intervals = []
        for i, (start, label) in enumerate(rhythm_events):
            end    = rhythm_events[i + 1][0] if i + 1 < len(rhythm_events) else n_samples
            mapped = RHYTHM_MAP.get(label)
            if mapped:
                intervals.append((start, end, mapped))

        if not intervals:
            print(f'  {rec}: no NSR/AFIB intervals — skipping')
            continue

        rec_counts = {'NSR': 0, 'AFIB': 0}

        for start, end, label in intervals:
            start     = max(start, 0)
            end       = min(end, n_samples)
            n_windows = (end - start) // WIN_LEN

            for w in range(n_windows):
                seg_start   = start + w * WIN_LEN
                seg_end     = seg_start + WIN_LEN
                window_syms = [sym for s, sym in beat_index
                               if seg_start <= s < seg_end]
                if label == 'NSR':
                    if not window_syms or not all(
                            sym in NORMAL_BEAT_SYMBOLS for sym in window_syms):
                        continue
                elif label == 'AFIB':
                    if any(sym in AFIB_EXCL_SYMBOLS for sym in window_syms):
                        continue

                segment = signal[seg_start:seg_end]
                if len(segment) != WIN_LEN:
                    continue

                fname = f'{rec}_{label}_{rec_counts[label]:05d}.npy'
                np.save(os.path.join(NSR_AFIB_DIR, label, fname), segment)
                rec_counts[label] += 1

        for lbl in ('NSR', 'AFIB'):
            total_counts[lbl] += rec_counts[lbl]

        summary = ', '.join(f'{k}={v}' for k, v in rec_counts.items() if v > 0)
        print(f'  {rec} (ch{ch}=MLII): {summary or "no qualifying windows"}')

    print('\n=== Block 2 complete ===')
    for lbl, count in total_counts.items():
        print(f'  {lbl}: {count:,} segments saved')
    _mark_done('block2')

---
## Block 3 — Extract PVC / LBBB Segments

Streams MIT-BIH records from PhysioNet (`mitdb`).

- `V` → PVC, `L` → LBBB beat annotations mark the R-peak.
- Window placement: PVC — R-peak jittered (40 % centred at sample 512; 60 % uniform in [205, 819]). LBBB — always centred at sample 512.
- Windows rejected if any beat other than Normal (`N`, `.`) or the same class is present, or if the window exceeds the signal boundary.

Output: `pvc-lbbb/PVC/`, `pvc-lbbb/LBBB/`

In [ ]:
_STEP = 'block3'
if _is_done(_STEP):
    print(f'✓ Block 3 already done — skipping.')
else:
    LABEL_MAP = {'V': 'PVC', 'L': 'LBBB'}
    NORMAL_BEATS = {'N', '.'}
    ALLOWED_IN_WINDOW = {
        'V': NORMAL_BEATS | {'V'},
        'L': NORMAL_BEATS | {'L'},
    }
    OFFSET_CENTERED_RATIO = 0.40
    OFFSET_MIN_POS        = int(0.20 * WIN_LEN)   # 205
    OFFSET_MAX_POS        = int(0.80 * WIN_LEN)   # 819
    _rng_window_pos       = random.Random(42)

    def _choose_target_pos():
        if _rng_window_pos.random() < OFFSET_CENTERED_RATIO:
            return HALF_WIN
        return _rng_window_pos.randint(OFFSET_MIN_POS, OFFSET_MAX_POS)

    for folder in LABEL_MAP.values():
        p = os.path.join(PVC_LBBB_DIR, folder)
        shutil.rmtree(p, ignore_errors=True)
        os.makedirs(p, exist_ok=True)

    total_counts_pvclbbb = {folder: 0 for folder in LABEL_MAP.values()}

    for rec in mitdb_records:
        _r, _pn = _wfdb_args(MITDB_LOCAL, rec, 'mitdb')
        try:
            hdr = wfdb.rdheader(_r, pn_dir=_pn)
        except Exception as e:
            print(f'  [SKIP] {rec}: cannot read header — {e}')
            continue

        ch = get_mlii_channel(hdr.sig_name)
        if ch is None:
            print(f'  [SKIP] {rec}: no MLII channel')
            continue

        try:
            signal_obj = wfdb.rdrecord(_r, channels=[ch], pn_dir=_pn)
            signal = signal_obj.p_signal[:, 0].astype(np.float32)
        except Exception as e:
            print(f'  [WARN] {rec}: cannot read signal — {e}')
            continue

        try:
            ann = wfdb.rdann(_r, 'atr', pn_dir=_pn)
        except Exception as e:
            print(f'  [WARN] {rec}: cannot read annotations — {e}')
            continue

        n_samples  = len(signal)
        beat_index = list(zip(ann.sample, ann.symbol))

        rec_counts = {folder: 0 for folder in LABEL_MAP.values()}

        for sample_idx, symbol in beat_index:
            if symbol not in LABEL_MAP:
                continue

            folder     = LABEL_MAP[symbol]
            target_pos = _choose_target_pos() if symbol == 'V' else HALF_WIN
            seg_start  = sample_idx - target_pos
            seg_end    = seg_start + WIN_LEN

            if seg_start < 0 or seg_end > n_samples:
                continue

            allowed      = ALLOWED_IN_WINDOW[symbol]
            window_beats = [sym for s, sym in beat_index if seg_start <= s < seg_end]
            if any(b not in allowed for b in window_beats):
                continue

            segment = signal[seg_start:seg_end]
            if len(segment) != WIN_LEN:
                continue

            fname = f'{rec}_{folder}_{rec_counts[folder]:05d}.npy'
            np.save(os.path.join(PVC_LBBB_DIR, folder, fname), segment)
            rec_counts[folder] += 1

        for folder in LABEL_MAP.values():
            total_counts_pvclbbb[folder] += rec_counts[folder]

        summary = ', '.join(f'{k}={v}' for k, v in rec_counts.items() if v > 0)
        print(f'  {rec} (ch{ch}=MLII): {summary or "no qualifying windows"}')

    print('\n=== Block 3 complete ===')
    for folder, count in total_counts_pvclbbb.items():
        print(f'  {folder}: {count:,} segments saved')
    _mark_done('block3')

---
## Block 3b — Extract NSR / AFIB / LBBB from PTB-XL

Streams records from PhysioNet (`ptb-xl/1.0.3`). Appends to Block 2 / 3 output folders.

- Only records with a single dominant SCP code ≥ 80 % confidence: `NORM` → NSR, `AFIB` → AFIB, `CLBBB` → LBBB.
- Records where any other target-class code reaches ≥ 40 % confidence are rejected.
- AFIB and NSR records are screened with PTB-XL+ 12SL (`12sl_statements.csv`): records with `PVC`, `VPR`, `BIGU`, `TRIGU`, or `VESC` at ≥ 50 % confidence are rejected.
- Lead II (index 1) extracted from 500 Hz 12-lead recording, resampled to 360 Hz via `resample_poly(x, 18, 25)`.
- Each 10-second record yields ~3 non-overlapping 1 024-sample windows.
- `patient_id` embedded in filenames for patient-wise splitting.

Output: appends to `nsr-afib/NSR/`, `nsr-afib/AFIB/`, `pvc-lbbb/LBBB/`

In [ ]:
_STEP = 'block3b'
if _is_done(_STEP):
    print(f'✓ Block 3b already done — skipping.')
else:
    import ast, urllib.request, threading

    PTBXL_VERSION     = 'ptb-xl/1.0.3'
    PTBXLPLUS_VERSION  = 'ptb-xl-plus/1.0.1'
    PTBXL_RESAMP_UP   = 18
    PTBXL_RESAMP_DOWN = 25
    PTBXL_N_WORKERS   = 8
    PTBXL_LEAD_IDX    = 1
    PTBXL_MIN_CONF    = 80.0
    PTBXL_EXCL_CONF   = 40.0
    PTBXL_SCP_MAP     = {'NORM': 'NSR', 'AFIB': 'AFIB', 'CLBBB': 'LBBB'}

    # PTB-XL+ 12SL ectopy codes used to post-filter AFIB records.
    # Replaces the v2 SCP confidence heuristic with a direct algorithmic flag.
    PTBXLPLUS_ECTOPY_CODES = {'PVC', 'VPR', 'BIGU', 'TRIGU', 'VESC'}
    PTBXLPLUS_ECTOPY_CONF  = 50.0

    ptbxl_meta_dir = os.path.join(BASE_DIR, 'ptbxl_meta')
    os.makedirs(ptbxl_meta_dir, exist_ok=True)
    meta_csv_path = os.path.join(ptbxl_meta_dir, 'ptbxl_database.csv')

    if not os.path.exists(meta_csv_path):
        print('Downloading PTB-XL metadata CSV (~5 MB) ...')
        url = f'https://physionet.org/files/{PTBXL_VERSION}/ptbxl_database.csv'
        urllib.request.urlretrieve(url, meta_csv_path)
        print('Done.')
    else:
        print('PTB-XL metadata CSV already cached.')

    df_meta = pd.read_csv(meta_csv_path, index_col='ecg_id')
    df_meta['scp_codes'] = df_meta['scp_codes'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else {})
    print(f'PTB-XL records in metadata: {len(df_meta):,}')

    def get_ptbxl_label(scp_dict):
        """Return class label if exactly one target SCP code meets confidence threshold.

        Exclusion layer: cross-class contamination only.
        Ventricular ectopy guard is handled separately via PTB-XL+ 12SL post-filter.
        """
        hits = {k: v for k, v in scp_dict.items()
                if k in PTBXL_SCP_MAP and v >= PTBXL_MIN_CONF}
        if len(hits) != 1:
            return None
        winner = next(iter(hits))
        # Cross-class contamination
        for code, conf in scp_dict.items():
            if code != winner and code in PTBXL_SCP_MAP and conf >= PTBXL_EXCL_CONF:
                return None
        return PTBXL_SCP_MAP[winner]

    df_meta['ptbxl_class'] = df_meta['scp_codes'].apply(get_ptbxl_label)

    # --- PTB-XL+ 12SL ectopy guard -------------------------------------------
    # Download (or use local copy) of 12sl_statements.csv from PTB-XL+.
    # Any AFIB record where an ectopy code appears in statements_ext at
    # ≥ PTBXLPLUS_ECTOPY_CONF confidence is nulled out.
    _12sl_local  = os.path.join(TMP_RAW_DIR, 'ptb-xl-plus', 'labels', '12sl_statements.csv')
    _12sl_cached = os.path.join(ptbxl_meta_dir, '12sl_statements.csv')
    if os.path.exists(_12sl_local):
        _12sl_path = _12sl_local
        print('Using local PTB-XL+ 12sl_statements.csv')
    elif os.path.exists(_12sl_cached):
        _12sl_path = _12sl_cached
        print('PTB-XL+ 12sl_statements.csv already cached.')
    else:
        print('Downloading PTB-XL+ 12sl_statements.csv ...')
        _12sl_url = f'https://physionet.org/files/{PTBXLPLUS_VERSION}/labels/12sl_statements.csv'
        urllib.request.urlretrieve(_12sl_url, _12sl_cached)
        _12sl_path = _12sl_cached
        print('Done.')

    import ast as _ast12sl
    _df_12sl = pd.read_csv(_12sl_path, index_col='ecg_id')

    def _parse_stmt_ext(x):
        if not isinstance(x, str) or x.strip() in ('', '[]'):
            return []
        try:
            return _ast12sl.literal_eval(x)
        except Exception:
            return []

    _df_12sl['_parsed'] = _df_12sl['statements_ext'].apply(_parse_stmt_ext)
    _ptbxlp_ectopy_ids = {
        idx for idx, row in _df_12sl.iterrows()
        if any(code in PTBXLPLUS_ECTOPY_CODES and conf >= PTBXLPLUS_ECTOPY_CONF
               for code, conf in row['_parsed'])
    }
    print(f'PTB-XL+ 12SL guard: {len(_ptbxlp_ectopy_ids):,} records flagged for ectopy')

    _n_guarded_before = df_meta['ptbxl_class'].isin(['AFIB', 'NSR']).sum()
    df_meta.loc[
        df_meta.index.isin(_ptbxlp_ectopy_ids) & (df_meta['ptbxl_class'].isin(['AFIB', 'NSR'])),
        'ptbxl_class'
    ] = None
    _n_guarded_after = df_meta['ptbxl_class'].isin(['AFIB', 'NSR']).sum()
    print(f'12SL guard removed {_n_guarded_before - _n_guarded_after:,} AFIB/NSR records with ectopy')
    # -------------------------------------------------------------------------

    df_target = df_meta[df_meta['ptbxl_class'].notna()].copy()

    print('\nRecords passing single-label filter:')
    for lbl, cnt in df_target['ptbxl_class'].value_counts().items():
        print(f'  {lbl}: {cnt:,} records  (~{cnt * 3:,} segments at 360 Hz / 10 s)')

    ptbxl_counts   = {cls: 0 for cls in PTBXL_SCP_MAP.values()}
    n_errors_ptbxl = 0
    _lock          = threading.Lock()

    def _process_ptbxl(ecg_id, row):
        cls        = row['ptbxl_class']
        patient_id = int(row['patient_id'])
        filename_hr = row['filename_hr']
        rec_subdir  = '/'.join(filename_hr.split('/')[:-1])
        rec_name    = filename_hr.split('/')[-1]
        if PTBXL_LOCAL_OK:
            _ptbxl_rec = os.path.join(PTBXL_LOCAL, rec_subdir, rec_name)
            _ptbxl_pn  = None
        else:
            _ptbxl_rec = rec_name
            _ptbxl_pn  = f'{PTBXL_VERSION}/{rec_subdir}'
        out_dir = os.path.join(
            NSR_AFIB_DIR if cls in ('NSR', 'AFIB') else PVC_LBBB_DIR, cls)

        try:
            rec_obj    = wfdb.rdrecord(_ptbxl_rec, pn_dir=_ptbxl_pn,
                                       channels=[PTBXL_LEAD_IDX])
            signal_raw = rec_obj.p_signal[:, 0].astype(np.float64)
        except Exception as e:
            return (cls, 0, str(e))

        signal_360 = sp_signal.resample_poly(
            signal_raw, PTBXL_RESAMP_UP, PTBXL_RESAMP_DOWN
        ).astype(np.float32)

        n_saved = 0
        for w in range(len(signal_360) // WIN_LEN):
            seg = signal_360[w * WIN_LEN : (w + 1) * WIN_LEN]
            if len(seg) != WIN_LEN:
                continue
            fname = f'ptbxl_{patient_id:05d}_{ecg_id:05d}_w{w:02d}_{cls}.npy'
            np.save(os.path.join(out_dir, fname), seg)
            n_saved += 1
        return (cls, n_saved, None)

    rows    = list(df_target.iterrows())
    n_total = len(rows)
    n_done  = 0

    with ThreadPoolExecutor(max_workers=PTBXL_N_WORKERS) as pool:
        futures = {pool.submit(_process_ptbxl, eid, r): eid for eid, r in rows}
        for fut in as_completed(futures):
            cls, n_segs, err = fut.result()
            with _lock:
                if err:
                    n_errors_ptbxl += 1
                    if n_errors_ptbxl <= 3:
                        print(f'  [WARN] ecg_id={futures[fut]}: {err}')
                else:
                    ptbxl_counts[cls] += n_segs
                n_done += 1
                if n_done % 500 == 0:
                    pct = n_done / n_total * 100
                    tot = sum(ptbxl_counts.values())
                    print(f'  {n_done}/{n_total} ({pct:.0f}%) — {tot:,} segments ...')

    print(f'\n=== Block 3b complete ===')
    print(f'  Errors / skipped : {n_errors_ptbxl}')
    for cls_name, cnt in ptbxl_counts.items():
        print(f'  {cls_name}: {cnt:,} segments appended')
    _mark_done('block3b')

---
## Block 3c — Extract AFIB / PVC from Long-Term AF Database (LTAFDB)

Streams 84 long-term records from PhysioNet (`ltafdb/1.0.0`). Appends to Block 2 / 3 output folders.

- Sampling rate 128 Hz, resampled to 360 Hz via `resample_poly(x, 45, 16)`. Channel 0 used.
- **AFIB windows:** rhythm must be `(AFIB`; rejected if any ventricular/ectopy beat is present. Non-overlapping 1 024-sample windows, capped at 2 000/record.
- **PVC windows:** `V` beats in non-AFIB rhythm intervals; purity `{N, ., V}`; R-peak jittered (40 % centred at 512, 60 % uniform in [205, 819]); capped at 200/record.

Output: appends to `nsr-afib/AFIB/`, `pvc-lbbb/PVC/`

In [ ]:
_STEP = 'block3c'
if _is_done(_STEP):
    print(f'✓ Block 3c already done — skipping.')
else:
    LTAFDB_PNDIR       = LTAFDB_PN
    LTAFDB_RESAMP_UP   = 45
    LTAFDB_RESAMP_DOWN = 16
    LTAFDB_RHYTHM_MAP  = {
        '(AFIB': 'AFIB',
    }

    LTAFDB_ECTOPIC_SYMS  = {'V', 'r', 'E', 'F', 'L', 'R', 'B', '!', '[', ']'}
    LTAFDB_MAX_PER_REC   = 2_000
    _LTAFDB_SCAN_LIMIT   = LTAFDB_MAX_PER_REC * 2

    import bisect as _bisect
    LTAFDB_PVC_MAX_PER_REC  = 200
    _LTAFDB_PVC_SCAN_LIMIT  = LTAFDB_PVC_MAX_PER_REC * 2
    _LTAFDB_PVC_ALLOWED     = frozenset({'N', '.', 'V'})
    _LTAF_PVC_CENTERED_RATIO = 0.40
    _LTAF_PVC_MIN_POS        = int(0.20 * WIN_LEN)   # 205
    _LTAF_PVC_MAX_POS        = int(0.80 * WIN_LEN)   # 819
    _rng_ltaf_pvc            = random.Random(99)

    def _ltaf_pvc_target_pos():
        if _rng_ltaf_pvc.random() < _LTAF_PVC_CENTERED_RATIO:
            return HALF_WIN
        return _rng_ltaf_pvc.randint(_LTAF_PVC_MIN_POS, _LTAF_PVC_MAX_POS)

    print('Fetching LTAFDB record list ...')
    if os.path.isdir(LTAFDB_PNDIR):
        ltafdb_records = sorted(
            os.path.splitext(f)[0] for f in os.listdir(LTAFDB_PNDIR) if f.endswith('.hea')
        )
        print(f'Records (local): {ltafdb_records}\n')
    else:
        ltafdb_records = wfdb.get_record_list('ltafdb')
        print(f'Records (PhysioNet): {ltafdb_records}\n')

    ltafdb_counts   = {'AFIB': 0, 'PVC': 0}
    n_errors_ltafdb = 0

    for rec in ltafdb_records:
        _r, _pn = _wfdb_args(LTAFDB_LOCAL, rec, 'ltafdb')
        try:
            hdr = wfdb.rdheader(_r, pn_dir=_pn)
        except Exception as e:
            print(f'  [SKIP] {rec}: header error — {e}')
            continue

        try:
            rec_obj    = wfdb.rdrecord(_r, channels=[0], pn_dir=_pn)
            signal_raw = rec_obj.p_signal[:, 0].astype(np.float64)
        except Exception as e:
            print(f'  [WARN] {rec}: signal error — {e}')
            n_errors_ltafdb += 1
            continue

        try:
            ann = wfdb.rdann(_r, 'atr', pn_dir=_pn)
        except Exception as e:
            print(f'  [WARN] {rec}: annotation error — {e}')
            n_errors_ltafdb += 1
            continue

        signal_360    = sp_signal.resample_poly(
            signal_raw, LTAFDB_RESAMP_UP, LTAFDB_RESAMP_DOWN
        ).astype(np.float32)
        n_samples_360 = len(signal_360)
        scale         = LTAFDB_RESAMP_UP / LTAFDB_RESAMP_DOWN

        beat_index_360 = [(int(s * scale), sym)
                          for s, sym in zip(ann.sample, ann.symbol)]

        rhythm_events = []
        for s_raw, sym, aux in zip(ann.sample, ann.symbol, ann.aux_note):
            if sym == '+':
                label = aux.strip().rstrip('\x00').strip()
                rhythm_events.append((int(s_raw * scale), label))

        if not rhythm_events:
            print(f'  {rec}: no rhythm annotations — skipping')
            continue

        # Full interval list for PVC rhythm-guard bisect lookup (all rhythms)
        _all_intervals  = []
        for _ai, (_rs, _rl) in enumerate(rhythm_events):
            _re = rhythm_events[_ai + 1][0] if _ai + 1 < len(rhythm_events) else n_samples_360
            _all_intervals.append((_rs, _re, _rl))
        _all_int_starts = [iv[0] for iv in _all_intervals]

        intervals = []
        for i, (start, label) in enumerate(rhythm_events):
            end    = rhythm_events[i+1][0] if i+1 < len(rhythm_events) else n_samples_360
            mapped = LTAFDB_RHYTHM_MAP.get(label)
            if mapped:
                intervals.append((start, end, mapped))

        rec_counts = {'AFIB': 0, 'PVC': 0}

        # Collect valid AFIB segment start offsets, stopping early once we
        # have 2x the cap (sufficient for a representative shuffle).
        _afib_starts = []
        _scan_done   = False
        for start, end, label in intervals:
            if _scan_done:
                break
            start = max(start, 0)
            end   = min(end, n_samples_360)
            for w in range((end - start) // WIN_LEN):
                seg_start  = start + w * WIN_LEN
                seg_end    = seg_start + WIN_LEN
                if seg_end > n_samples_360:
                    continue
                window_syms = [sym for s, sym in beat_index_360
                               if seg_start <= s < seg_end]
                if not any(sym in LTAFDB_ECTOPIC_SYMS for sym in window_syms):
                    _afib_starts.append(seg_start)
                    if len(_afib_starts) >= _LTAFDB_SCAN_LIMIT:
                        _scan_done = True
                        break

        # Shuffle and cap
        _rng_ltaf = np.random.default_rng(42)
        if len(_afib_starts) > LTAFDB_MAX_PER_REC:
            _afib_starts = list(_rng_ltaf.choice(
                _afib_starts, size=LTAFDB_MAX_PER_REC, replace=False))

        for _ss in _afib_starts:
            seg   = signal_360[_ss : _ss + WIN_LEN]
            fname = f'ltafdb_{rec}_AFIB_{rec_counts["AFIB"]:05d}.npy'
            np.save(os.path.join(NSR_AFIB_DIR, 'AFIB', fname), seg)
            rec_counts['AFIB'] += 1

        # ── Collect PVC windows (non-AFIB intervals, jittered) ──────────────
        _pvc_starts = []
        _pvc_done   = False
        for _bs, _bsym in beat_index_360:
            if _pvc_done:
                break
            if _bsym != 'V':
                continue
            # Reject beats inside AFIB intervals
            _bi = _bisect.bisect_right(_all_int_starts, _bs) - 1
            if _bi < 0 or _all_intervals[_bi][2] == '(AFIB':
                continue
            # Jittered window (same policy as v7 Block 3 PVC)
            _tgt   = _ltaf_pvc_target_pos()
            _seg_s = _bs - _tgt
            _seg_e = _seg_s + WIN_LEN
            if _seg_s < 0 or _seg_e > n_samples_360:
                continue
            # Window purity: all beats must be N, ., or V
            _wb = [sym for s, sym in beat_index_360 if _seg_s <= s < _seg_e]
            if not _wb or not all(sym in _LTAFDB_PVC_ALLOWED for sym in _wb):
                continue
            _pvc_starts.append(_seg_s)
            if len(_pvc_starts) >= _LTAFDB_PVC_SCAN_LIMIT:
                _pvc_done = True

        # Shuffle and cap — PVC
        if len(_pvc_starts) > LTAFDB_PVC_MAX_PER_REC:
            _pvc_starts = _rng_ltaf_pvc.sample(_pvc_starts, LTAFDB_PVC_MAX_PER_REC)
        for _ss in _pvc_starts:
            seg   = signal_360[_ss : _ss + WIN_LEN]
            fname = f'ltafdb_{rec}_PVC_{rec_counts["PVC"]:05d}.npy'
            np.save(os.path.join(PVC_LBBB_DIR, 'PVC', fname), seg)
            rec_counts['PVC'] += 1

        ltafdb_counts['AFIB'] += rec_counts['AFIB']
        ltafdb_counts['PVC']  += rec_counts['PVC']

        summary = ', '.join(f'{k}={v}' for k, v in rec_counts.items() if v > 0)
        print(f'  {rec}: {summary or "no qualifying windows"}')

    print(f'\n=== Block 3c complete (LTAFDB) ===')
    print(f'  Errors / skipped : {n_errors_ltafdb}')
    for lbl, cnt in ltafdb_counts.items():
        print(f'  {lbl}: {cnt:,} segments appended')
    _mark_done('block3c')

---
## Block 3d — Extract PVC / LBBB from INCART 12-Lead Database

Streams 75 records from PhysioNet (`incartdb`). Appends to Block 3 output folders.

- Sampling rate 257 Hz, resampled to 360 Hz via `resample_poly(x, 360, 257)`. Lead II used where available; falls back to channel 0.
- `V` → PVC, `L` → LBBB. Window placement identical to Block 3: PVC jittered (40 % centred, 60 % uniform in [205, 819]); LBBB always centred at 512.
- Same purity check as Block 3.

Output: appends to `pvc-lbbb/PVC/`, `pvc-lbbb/LBBB/`

In [ ]:
_STEP = 'block3d'
if _is_done(_STEP):
    print(f'✓ Block 3d already done — skipping.')
else:
    INCART_PNDIR       = INCART_PN
    INCART_RESAMP_UP   = 360
    INCART_RESAMP_DOWN = 257
    INCART_LABEL_MAP   = {'V': 'PVC', 'L': 'LBBB'}
    INCART_NORMAL      = {'N', '.'}
    INCART_ALLOWED     = {
        'V': INCART_NORMAL | {'V'},
        'L': INCART_NORMAL | {'L'},
    }
    INCART_OFFSET_CENTERED_RATIO = 0.40
    INCART_OFFSET_MIN_POS        = int(0.20 * WIN_LEN)   # 205
    INCART_OFFSET_MAX_POS        = int(0.80 * WIN_LEN)   # 819
    _incart_rng_window_pos       = random.Random(84)

    def _choose_incart_target_pos():
        if _incart_rng_window_pos.random() < INCART_OFFSET_CENTERED_RATIO:
            return HALF_WIN
        return _incart_rng_window_pos.randint(INCART_OFFSET_MIN_POS, INCART_OFFSET_MAX_POS)

    def get_lead2_channel(sig_names):
        """Return index of Lead II, or 0 as fallback."""
        for i, n in enumerate(sig_names):
            if n.strip().upper() in ('II', 'MLII', 'ML II', 'LEAD II'):
                return i
        return 0

    print('Fetching INCART record list ...')
    if os.path.isdir(INCART_PNDIR):
        incart_records = sorted(
            os.path.splitext(f)[0] for f in os.listdir(INCART_PNDIR) if f.endswith('.hea')
        )
        print(f'Records: {len(incart_records)} (local)\n')
    else:
        incart_records = wfdb.get_record_list(INCART_PNDIR)
        print(f'Records: {len(incart_records)} ({incart_records[0]} — {incart_records[-1]}) (PhysioNet)\n')

    incart_counts   = {'PVC': 0, 'LBBB': 0}
    n_errors_incart = 0

    for rec in incart_records:
        _r, _pn = _wfdb_args(INCART_LOCAL, rec, 'incartdb')
        try:
            hdr = wfdb.rdheader(_r, pn_dir=_pn)
        except Exception as e:
            print(f'  [SKIP] {rec}: header error — {e}')
            n_errors_incart += 1; continue

        ch = get_lead2_channel(hdr.sig_name)

        try:
            rec_obj    = wfdb.rdrecord(_r, channels=[ch], pn_dir=_pn)
            signal_raw = rec_obj.p_signal[:, 0].astype(np.float64)
        except Exception as e:
            print(f'  [WARN] {rec}: signal error — {e}')
            n_errors_incart += 1; continue

        try:
            ann = wfdb.rdann(_r, 'atr', pn_dir=_pn)
        except Exception as e:
            print(f'  [WARN] {rec}: annotation error — {e}')
            n_errors_incart += 1; continue

        signal_360     = sp_signal.resample_poly(
            signal_raw, INCART_RESAMP_UP, INCART_RESAMP_DOWN
        ).astype(np.float32)
        n_samples_360  = len(signal_360)
        scale          = INCART_RESAMP_UP / INCART_RESAMP_DOWN

        beat_index_360 = [(int(s * scale), sym)
                          for s, sym in zip(ann.sample, ann.symbol)]

        rec_counts = {'PVC': 0, 'LBBB': 0}

        for sample_360, symbol in beat_index_360:
            if symbol not in INCART_LABEL_MAP:
                continue

            folder     = INCART_LABEL_MAP[symbol]
            target_pos = _choose_incart_target_pos() if symbol == 'V' else HALF_WIN
            seg_start  = sample_360 - target_pos
            seg_end    = seg_start + WIN_LEN
            if seg_start < 0 or seg_end > n_samples_360:
                continue

            allowed      = INCART_ALLOWED[symbol]
            window_beats = [sym for s, sym in beat_index_360
                            if seg_start <= s < seg_end]
            if any(b not in allowed for b in window_beats):
                continue

            seg = signal_360[seg_start:seg_end]
            if len(seg) != WIN_LEN:
                continue

            fname = f'incart_{rec}_{folder}_{rec_counts[folder]:05d}.npy'
            np.save(os.path.join(PVC_LBBB_DIR, folder, fname), seg)
            rec_counts[folder] += 1

        for lbl in ('PVC', 'LBBB'):
            incart_counts[lbl] += rec_counts[lbl]

        summary = ', '.join(f'{k}={v}' for k, v in rec_counts.items() if v > 0)
        print(f'  {rec} (ch{ch}={hdr.sig_name[ch]}): {summary or "no qualifying windows"}')

    print(f'\n=== Block 3d complete ===')
    print(f'  Errors / skipped : {n_errors_incart}')
    for lbl, cnt in incart_counts.items():
        print(f'  {lbl}: {cnt:,} segments appended')
    _mark_done('block3d')


_SEAL_STEP = 'seal_segs'
if _is_done(_SEAL_STEP):
    print('✓ segs.tar.gz already saved on Drive.')
elif all(_is_done(s) for s in _EXTRACTION_STEPS):
    print('All extraction blocks done — sealing segments to Drive ...')
    _compress_to_drive(
        os.path.join(TMP_BASE, 'segs'),
        SEGS_TAR_DRIVE,
        arc_name='segs',
    )
    _mark_done(_SEAL_STEP)
    print('✓ segs.tar.gz written.')
else:
    _remaining = [s for s in _EXTRACTION_STEPS if not _is_done(s)]
    print(f'[SKIP seal] Still waiting for: {_remaining}')

---
## Block 4 — Patient-wise Train / Val / Test Split

Splits at the record level — all segments from a record go to the same partition.

- Records grouped by class signature and database; distributed ~75 % train / 12.5 % val / 12.5 % test.
- LBBB-only and AFIB-only records guaranteed ≥ 1 in val and test.
- Per-class segment cap applied per split to prevent any single source from dominating.

Output: `split/record_split.json`, `split/train.csv`, `split/val.csv`, `split/test.csv`

In [ ]:
import subprocess, time as _time

try:
    subprocess.run(['fusermount', '-uz', '/content/drive'], capture_output=True)
    _time.sleep(3)
    from google.colab import drive as _drive
    _drive.mount('/content/drive', force_remount=True)
    print('Waiting for Drive FUSE to stabilise', end='', flush=True)
    for _probe in range(30):
        try:
            os.listdir(BASE_DIR)
            print(f'  OK ({_probe * 2}s)')
            break
        except OSError:
            print('.', end='', flush=True)
            _time.sleep(2)
    else:
        print('  [WARN] FUSE still not responding after 60 s — proceeding anyway')
    print('Drive remounted.')
except Exception as _e:
    print(f'[WARN] Drive remount failed: {_e}')

os.makedirs(SPLIT_DIR, exist_ok=True)

FOLDERS = {
    'PVC':  os.path.join(PVC_LBBB_DIR, 'PVC'),
    'LBBB': os.path.join(PVC_LBBB_DIR, 'LBBB'),
    'NSR':  os.path.join(NSR_AFIB_DIR, 'NSR'),
    'AFIB': os.path.join(NSR_AFIB_DIR, 'AFIB'),
}

def get_record_id(fname):
    """Patient/record-level ID for patient-wise splitting."""
    parts = os.path.splitext(fname)[0].split('_')
    if parts[0] == 'ptbxl':
        return f'ptbxl_p{parts[1]}'
    if parts[0] == 'ltafdb':
        return f'ltafdb_{parts[1]}'
    if parts[0] == 'incart':
        return f'incart_{parts[1]}'
    return parts[0]

rec_files = defaultdict(lambda: defaultdict(list))

for cls, folder in FOLDERS.items():
    for _attempt in range(4):
        try:
            _fnames = sorted(os.listdir(folder))
            break
        except OSError:
            if _attempt == 3:
                raise
            print(f'  [WARN] FUSE I/O error on {folder}, retry {_attempt+1}/3...')
            _time.sleep(5)
    for fname in _fnames:
        if not fname.endswith('.npy'):
            continue
        rec = get_record_id(fname)
        rec_files[rec][cls].append(os.path.join(folder, fname))

all_records = sorted(rec_files.keys())
print(f'Total records found: {len(all_records)}')
for rec in all_records:
    counts = {c: len(v) for c, v in rec_files[rec].items()}
    print(f'  {rec}: {counts}')


def distribute(records, target=(0.75, 0.125, 0.125)):
    """Return (train, val, test) lists for a group of records."""
    n = len(records)
    if n == 1:
        return records, [], []
    if n == 2:
        return [records[0]], [records[1]], []
    if n == 3:
        return [records[0]], [records[1]], [records[2]]
    n_val   = max(1, round(n * target[1]))
    n_test  = max(1, round(n * target[2]))
    n_train = n - n_val - n_test
    return records[:n_train], records[n_train:n_train+n_val], records[n_train+n_val:]

def get_db_prefix(rec):
    if rec.startswith('ptbxl_'): return 'ptbxl'
    if rec.startswith('ltafdb_'): return 'ltafdb'
    if rec.startswith('incart_'):return 'incart'
    return 'mitbih'

groups = defaultdict(list)
for rec in all_records:
    sig = (get_db_prefix(rec), frozenset(rec_files[rec].keys()))
    groups[sig].append(rec)

print('Record groups by (database, class signature):')
for sig, recs in sorted(groups.items(), key=lambda x: len(x[1])):
    print(f'  {sig[0]} {set(sig[1])}: {len(recs)} records')

record_split = {}
for sig, recs in groups.items():
    train_r, val_r, test_r = distribute(recs)
    for r in train_r: record_split[r] = 'train'
    for r in val_r:   record_split[r] = 'val'
    for r in test_r:  record_split[r] = 'test'

split_classes = {'train': set(), 'val': set(), 'test': set()}
for rec, split in record_split.items():
    for cls in rec_files[rec]:
        split_classes[split].add(cls)

print('\nClasses in each split:')
for split, clses in split_classes.items():
    print(f'  {split}: {sorted(clses)}')


MAX_PER_CLASS = {'train': 30_000, 'val': 2_000, 'test': 2_000}

json_path = os.path.join(SPLIT_DIR, 'record_split.json')
with open(json_path, 'w') as f:
    json.dump(record_split, f, indent=2, sort_keys=True)
print(f'Saved: {json_path}')

split_data = {'train': [], 'val': [], 'test': []}
for rec in all_records:
    split = record_split[rec]
    for cls, paths in rec_files[rec].items():
        for p in paths:
            split_data[split].append((p, cls, rec))

import random as _random
_random.seed(42)
for split in ('train', 'val', 'test'):
    cap    = MAX_PER_CLASS[split]
    _by_cls = {}
    for row in split_data[split]:
        _by_cls.setdefault(row[1], []).append(row)
    _capped = []
    for cls, rows in _by_cls.items():
        if len(rows) > cap:
            rows = _random.sample(rows, cap)
            print(f'  [cap {split}] {cls}: capped to {cap:,}')
        _capped.extend(rows)
    split_data[split] = _capped

for split, rows in split_data.items():
    csv_path = os.path.join(SPLIT_DIR, f'{split}.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['filepath', 'class', 'record'])
        writer.writerows(rows)
    print(f'  {split}.csv : {len(rows):,} segments')

print('\n=== Block 4 complete ===')
total_all = sum(len(v) for v in split_data.values())
if total_all == 0:
    raise RuntimeError(
        'No segments found in any split.\n'
        'The extraction blocks (2, 3, 3b, 3c, 3d) have not been run this session '
        'and segs.tar.gz was not restored.\n'
        'Re-run the extraction blocks (or restore from Drive) before Block 4.'
    )
for split, rows in split_data.items():
    print(f'  {split:5s}: {len(rows):6,}  ({len(rows)/total_all*100:.1f}%)')

---
## Block 5 — Augment Training Set with NSTDB Noise

Noise recordings (BW, EM, MA) streamed from PhysioNet (`nstdb`).

- Equal quota per class selected for augmentation: `total_clean // (4 variants × n_classes)`, giving a ~1:1 clean-to-augmented ratio.
- Each selected segment produces 4 noisy variants: BW, EM, MA, ALL (combined), at a random SNR in [5, 10] dB.
- Noise windows sampled only from the top 70 % by windowed RMS.
- Val and test CSVs copied unchanged.

Output: `split-augmented/augmented/` + `split-augmented/{train,val,test}.csv`

In [ ]:
def load_noise_recording(rec_name, pn_dir='nstdb', channel=0):
    """Stream a NSTDB noise recording from PhysioNet."""
    rec = wfdb.rdrecord(rec_name, pn_dir=pn_dir)
    return rec.p_signal[:, channel].astype(np.float64)

def build_active_windows(noise_signal, win_len, percentile):
    """Return start indices of windows whose RMS exceeds the given percentile."""
    n_windows  = len(noise_signal) // win_len
    rms_values = np.array([
        np.sqrt(np.mean(noise_signal[i*win_len:(i+1)*win_len] ** 2))
        for i in range(n_windows)
    ])
    threshold = np.percentile(rms_values, percentile)
    return [i * win_len for i in range(n_windows) if rms_values[i] > threshold]

def sample_noise_window(noise_signal, active_starts, win_len, rng):
    """Pick one active window at random."""
    start = rng.choice(active_starts)
    return noise_signal[start : start + win_len].copy()

def scale_noise_to_snr(clean, noise, snr_db):
    """Scale noise so SNR(clean, noise) == snr_db."""
    rms_clean = np.sqrt(np.mean(clean ** 2))
    rms_noise = np.sqrt(np.mean(noise ** 2))
    if rms_noise < 1e-12 or rms_clean < 1e-12:
        return noise
    target_rms = rms_clean / (10 ** (snr_db / 20.0))
    return noise * (target_rms / rms_noise)

def scale_noise_to_snr_combined(clean, noise_list, snr_db):
    """Scale multiple noise components so their combined power hits snr_db."""
    n         = len(noise_list)
    rms_clean = np.sqrt(np.mean(clean ** 2))
    if rms_clean < 1e-12:
        return noise_list
    target_combined_rms = rms_clean / (10 ** (snr_db / 20.0))
    target_each_rms     = target_combined_rms / np.sqrt(n)
    scaled = []
    for noise in noise_list:
        rms_noise = np.sqrt(np.mean(noise ** 2))
        scaled.append(noise if rms_noise < 1e-12 else noise * (target_each_rms / rms_noise))
    return scaled


AUG_FRACTION              = 0.25
SNR_MIN_DB                = 5.0
SNR_MAX_DB                = 10.0
NOISE_ACTIVITY_PERCENTILE = 30
NOISE_TYPES               = ['BW', 'EM', 'MA']
NOISE_RECORD_NAMES        = {'BW': 'bw', 'EM': 'em', 'MA': 'ma'}
AUG_SEED                  = 42

_STEP = 'block5'
if _is_done(_STEP):
    print('✓ Block 5 already done — skipping.')
    noise_signals      = {}
    active_windows_map = {}
else:
    os.makedirs(AUG_DIR, exist_ok=True)

    print('Loading NSTDB noise recordings from PhysioNet ...')
    noise_signals      = {}
    active_windows_map = {}

    for ntype, rec_name in NOISE_RECORD_NAMES.items():
        sig = load_noise_recording(rec_name, pn_dir='nstdb')
        noise_signals[ntype] = sig
        aw  = build_active_windows(sig, WIN_LEN, NOISE_ACTIVITY_PERCENTILE)
        active_windows_map[ntype] = aw
        total_windows = len(sig) // WIN_LEN
        print(f'  {ntype}: {len(sig):,} samples, {total_windows} windows, '
              f'{len(aw)} active (>{NOISE_ACTIVITY_PERCENTILE}th pct RMS)')

    print('Noise recordings loaded.')

    rng_aug = random.Random(AUG_SEED)
    np.random.seed(AUG_SEED)

    if os.path.isdir(AUG_SEGS_DIR):
        shutil.rmtree(AUG_SEGS_DIR)
    os.makedirs(AUG_SEGS_DIR)

    for split in ('val', 'test'):
        src = os.path.join(SPLIT_DIR, f'{split}.csv')
        dst = os.path.join(AUG_DIR,   f'{split}.csv')
        shutil.copy2(src, dst)
    print('Copied val.csv and test.csv unchanged.')

    with open(os.path.join(SPLIT_DIR, 'train.csv'), newline='') as f:
        train_rows = list(csv.DictReader(f))

    _rows_by_cls = {}
    for _i, _r in enumerate(train_rows):
        _rows_by_cls.setdefault(_r['class'], []).append(_i)

    n_train       = len(train_rows)
    n_classes_aug = len(_rows_by_cls)
    quota_per_cls = n_train // (4 * n_classes_aug)

    print(f'Training segments  : {n_train:,}')
    print(f'Classes            : {n_classes_aug}')
    print(f'Quota per class    : {quota_per_cls:,} segs × 4 variants = {quota_per_cls*4:,} new rows')
    _total_new = quota_per_cls * 4 * n_classes_aug
    print(f'Total new rows     : {_total_new:,}  (ratio {_total_new/n_train:.2f}:1 vs clean)\n')

    aug_indices = set()
    for _cls in sorted(_rows_by_cls):
        _indices  = _rows_by_cls[_cls]
        _selected = rng_aug.sample(_indices, min(quota_per_cls, len(_indices)))
        aug_indices.update(_selected)
        print(f'  {_cls}: {len(_indices):,} clean  +  {len(_selected):,} selected')

    fieldnames = ['filepath', 'class', 'record']

    aug_order = [idx for idx in range(len(train_rows)) if idx in aug_indices]
    _pre = []
    for _idx in aug_order:
        _snr   = rng_aug.uniform(SNR_MIN_DB, SNR_MAX_DB)
        _nwins = {nt: sample_noise_window(noise_signals[nt],
                                          active_windows_map[nt],
                                          WIN_LEN, rng_aug)
                  for nt in NOISE_TYPES}
        _pre.append((_idx, _snr, _nwins))

    def _augment_one(idx, snr_db, noise_wins):
        row       = train_rows[idx]
        clean     = np.load(row['filepath']).astype(np.float64).flatten()[:WIN_LEN]
        base_name = os.path.splitext(os.path.basename(row['filepath']))[0]
        cls, record = row['class'], row['record']
        results   = []
        for ntype in NOISE_TYPES:
            scaled = scale_noise_to_snr(clean, noise_wins[ntype], snr_db)
            noisy  = (clean + scaled).astype(np.float32)
            np.save(os.path.join(AUG_SEGS_DIR, f'{base_name}_{ntype}.npy'), noisy)
            results.append({'filepath': os.path.join(AUG_SEGS_DIR, f'{base_name}_{ntype}.npy'),
                            'class': cls, 'record': record})
        noise_list  = [noise_wins[nt] for nt in NOISE_TYPES]
        scaled_list = scale_noise_to_snr_combined(clean, noise_list, snr_db)
        noisy_all   = (clean + sum(scaled_list)).astype(np.float32)
        np.save(os.path.join(AUG_SEGS_DIR, f'{base_name}_ALL.npy'), noisy_all)
        results.append({'filepath': os.path.join(AUG_SEGS_DIR, f'{base_name}_ALL.npy'),
                        'class': cls, 'record': record})
        return results

    new_rows = []
    n_done   = 0
    n_aug    = len(_pre)

    print(f'\nGenerating {n_aug * 4:,} augmented files to {AUG_SEGS_DIR} ...')
    _rows_by_source_idx = {}
    with ThreadPoolExecutor(N_WORKERS) as pool:
        futures = {pool.submit(_augment_one, _idx, _snr, _nwins): _idx
                   for _idx, _snr, _nwins in _pre}
        for fut in as_completed(futures):
            _src_idx = futures[fut]
            _rows_by_source_idx[_src_idx] = fut.result()
            n_done += 1
            if n_done % 500 == 0 or n_done == n_aug:
                print(f'  Augmented {n_done}/{n_aug} segments ...')

    for _idx in aug_order:
        new_rows.extend(_rows_by_source_idx[_idx])

    all_train_rows = train_rows + new_rows
    out_train_csv  = os.path.join(AUG_DIR, 'train.csv')
    with open(out_train_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_train_rows)

    print('\nSealing augmented segments to Drive ...')
    _compress_to_drive(AUG_SEGS_DIR, AUG_TAR_DRIVE, arc_name='aug_segs')

    print(f'\n=== Block 5 complete ===')
    print(f'  Original train rows : {len(train_rows):,}')
    print(f'  Augmented rows added: {len(new_rows):,}')
    print(f'  Total train rows    : {len(all_train_rows):,}')
    for _cls in sorted(_rows_by_cls):
        _clean_n = len(_rows_by_cls[_cls])
        _aug_n   = sum(1 for r in new_rows if r['class'] == _cls)
        print(f'  {_cls}: {_clean_n:,} clean + {_aug_n:,} aug = {_clean_n+_aug_n:,}')
    _mark_done(_STEP)

---
## Block 6 — Filter & Z-score Normalise

Applies signal processing to every segment in the augmented split.

1. Linear detrend
2. High-pass Butterworth (1 Hz, order 4)
3. Low-pass Butterworth (35 Hz, order 4)
4. Notch filter (61.7 Hz, Q=10)
5. Savitzky-Golay smoothing (window=15, poly=3)
6. Z-score normalisation

The three IIR filters are cascaded into a single SOS chain.

Output: `split-filtered/{train,val,test}.npz`

In [ ]:
HPF_CUTOFF_HZ = 1.0;   HPF_ORDER  = 4
LPF_CUTOFF_HZ = 35.0;  LPF_ORDER  = 4
NOTCH_FREQ_HZ = 61.7;  NOTCH_Q    = 10.0
SG_WINDOW     = 15;    SG_POLYORDER = 3

_sos_hpf   = sp_signal.butter(HPF_ORDER, HPF_CUTOFF_HZ,
                               btype='high', fs=SAMPLE_RATE_HZ, output='sos')
_sos_lpf   = sp_signal.butter(LPF_ORDER, LPF_CUTOFF_HZ,
                               btype='low',  fs=SAMPLE_RATE_HZ, output='sos')
_sos_notch = sp_signal.tf2sos(
                *sp_signal.iirnotch(NOTCH_FREQ_HZ, NOTCH_Q, SAMPLE_RATE_HZ))
_sos_all   = np.vstack([_sos_hpf, _sos_lpf, _sos_notch])

def filter_and_normalize(raw):
    """Detrend → HPF+LPF+notch (single SOS pass) → SavGol → z-score."""
    s = raw.astype(np.float64)
    s = sp_signal.detrend(s)
    s = sp_signal.sosfiltfilt(_sos_all, s)
    s = savgol_filter(s, SG_WINDOW, SG_POLYORDER)
    mean, std = np.mean(s), np.std(s) + 1e-8
    return ((s - mean) / std).astype(np.float32)

print('Filter coefficients built (combined SOS chain).')
print(f'  HPF  : {HPF_CUTOFF_HZ} Hz  order {HPF_ORDER}')
print(f'  LPF  : {LPF_CUTOFF_HZ} Hz  order {LPF_ORDER}')
print(f'  Notch: {NOTCH_FREQ_HZ} Hz  Q={NOTCH_Q}')
print(f'  SavGol: window={SG_WINDOW}, poly={SG_POLYORDER}')
print(f'  Workers: {N_WORKERS}')


_STEP = 'block6'
if _is_done(_STEP):
    print(f'✓ Block 6 already done — skipping.')
else:
    os.makedirs(FILTERED_DIR, exist_ok=True)

    def _filter_one(row):
        try:
            raw      = np.load(row['filepath'])
            filtered = filter_and_normalize(raw)
            if len(filtered) != WIN_LEN:
                filtered = filtered[:WIN_LEN] if len(filtered) > WIN_LEN \
                           else np.pad(filtered, (0, WIN_LEN - len(filtered)))
            return filtered, row['class'], None
        except Exception as e:
            return None, row['class'], e

    def process_split(split_name, in_dir):
        in_csv = os.path.join(in_dir, f'{split_name}.csv')
        with open(in_csv, newline='') as f:
            rows = list(csv.DictReader(f))

        total  = len(rows)
        errors = 0
        X_buf  = [None] * total
        y_buf  = np.full(total, -1, dtype=np.int32)
        print(f'\n[{split_name}]  {total:,} segments')

        with ThreadPoolExecutor(N_WORKERS) as pool:
            futures = {pool.submit(_filter_one, row): i for i, row in enumerate(rows)}
            done    = 0
            for fut in as_completed(futures):
                row_idx = futures[fut]
                signal, cls, err = fut.result()
                done += 1
                if err:
                    errors += 1
                else:
                    X_buf[row_idx] = signal
                    y_buf[row_idx] = CLASS_TO_IDX[cls]
                if done % 5000 == 0 or done == total:
                    print(f'    {done:,}/{total:,}  ({done/total*100:.1f}%)  errors={errors}')

        kept_indices = [i for i, sig in enumerate(X_buf) if sig is not None]
        if kept_indices:
            X = np.stack([X_buf[i] for i in kept_indices]).astype(np.float32, copy=False)
            y = np.array([y_buf[i] for i in kept_indices], dtype=np.int32)
        else:
            X = np.empty((0, WIN_LEN), dtype=np.float32)
            y = np.empty((0,), dtype=np.int32)

        filtered_rows = [rows[i] for i in kept_indices]
        os.makedirs(FILTERED_DIR, exist_ok=True)
        out_csv = os.path.join(FILTERED_DIR, f'{split_name}.csv')
        with open(out_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['filepath', 'class', 'record'])
            writer.writeheader()
            writer.writerows(filtered_rows)

        if len(filtered_rows) != len(X) or len(X) != len(y):
            raise RuntimeError(
                f'{split_name} alignment mismatch: csv={len(filtered_rows)} X={len(X)} y={len(y)}'
            )
        mismatched = sum(
            1 for i, r in enumerate(filtered_rows)
            if CLASS_TO_IDX[r['class']] != int(y[i])
        )
        if mismatched:
            raise RuntimeError(f'{split_name} label-index mismatch found in {mismatched} rows')

        print(f'  Collected {X.shape[0]:,} segments  ({errors} errors skipped)')
        return X, y, errors

    total_saved  = 0
    total_errors = 0
    for split in ('train', 'val', 'test'):
        X, y, errs = process_split(split, AUG_DIR)
        total_saved  += len(X)
        total_errors += errs

        tmp_path = f'/tmp/{split}.npz'
        npz_path = os.path.join(FILTERED_DIR, f'{split}.npz')
        print(f'  Saving {split}.npz → /tmp ({X.nbytes/1e6:.1f} MB uncompressed) ...')
        np.savez_compressed(tmp_path, X=X, y=y)
        _chk = np.load(tmp_path)
        assert _chk['X'].shape[0] == len(X), \
            f'{split} npz row count mismatch: {_chk["X"].shape[0]} vs {len(X)}'
        sz_mb = os.path.getsize(tmp_path) / 1e6
        print(f'  Copying {split}.npz to Drive ({sz_mb:.1f} MB compressed) ...')
        for _try in range(5):
            try:
                shutil.copy2(tmp_path, npz_path)
                break
            except OSError as _e:
                print(f'    [WARN] copy attempt {_try+1}/5 failed: {_e}')
                time.sleep(10)
        else:
            raise RuntimeError(f'Could not copy {tmp_path} to Drive after 5 attempts.')
        os.remove(tmp_path)
        print(f'  ✓ {split}.npz saved')

    print(f'\n=== Block 6 complete ===')
    print(f'  Total filtered segments : {total_saved:,}')
    print(f'  Total errors skipped    : {total_errors}')
    _mark_done(_STEP)

---
## Block 7 — Train Dual-Input 1-D CNN & Evaluate

> **GPU recommended** — `Runtime → Change runtime type → T4 GPU`

**Architecture:** Dual-input model combining a residual 1-D CNN signal branch with a hand-crafted RR-interval feature branch. The signal branch uses an initial convolution followed by three residual downsampling blocks, then a self-attention mechanism producing a morphology path (GlobalMaxPooling) and a temporal path (GlobalAveragePooling with dilated convolution). Both paths are concatenated and fused with the feature branch through a two-layer head.

**Training:**
- Loss: sparse categorical cross-entropy with inverse-frequency class weights
- Optimiser: Adam with ReduceLROnPlateau
- Early stopping on `val_macro_f1` with best-weights restoration
- Three independent runs with different random seeds

In [ ]:
BATCH_SIZE    = 64
EPOCHS        = 40
LEARNING_RATE = 5e-5

N_RUNS    = 3
RUN_SEEDS = [42, 0, 7]
assert len(RUN_SEEDS) == N_RUNS, 'RUN_SEEDS length must equal N_RUNS'

import datetime, json as _json, csv as _csv


def _extract_features(signals):
    """Compute RR interval statistics for each signal.
    Features: [mean_rr, std_rr, rmssd, pnn50, min_rr, max_rr, mean_qrs]
    """
    from scipy.signal import find_peaks, peak_widths
    features = []
    MIN_DIST = int(0.2 * 360)

    for i in range(len(signals)):
        sig     = signals[i].flatten()
        max_val = np.max(sig)
        if max_val < 0.1:
            features.append([0, 0, 0, 1.0, 0, 0, 0.0])
            continue

        peaks, _ = find_peaks(sig, height=max_val * 0.3, distance=MIN_DIST)

        if len(peaks) < 2:
            features.append([0, 0, 0, 1.0, 0, 0, 0.0])
            continue

        rrs      = np.diff(peaks) / 360.0
        mean_rr  = np.mean(rrs)
        std_rr   = np.std(rrs)
        min_rr   = np.min(rrs)
        max_rr   = np.max(rrs)
        diff_rrs = np.diff(rrs)
        rmssd    = np.sqrt(np.mean(diff_rrs**2)) if len(diff_rrs) > 0 else 0
        pnn50    = np.sum(np.abs(diff_rrs) > 0.050) / len(diff_rrs) if len(diff_rrs) > 0 else 0

        if len(peaks) > 0:
            widths, _, _, _ = peak_widths(sig, peaks, rel_height=0.5)
            mean_qrs = np.mean(widths) / 360.0
        else:
            mean_qrs = 0.0

        features.append([mean_rr, std_rr, rmssd, pnn50, min_rr, max_rr, mean_qrs])

    return np.array(features, dtype=np.float32)


def _load_split(split_name):
    npz_path = os.path.join(FILTERED_DIR, f'{split_name}.npz')
    print(f'  Loading {split_name}.npz ...')
    data  = np.load(npz_path)
    X_sig = data['X'].reshape(-1, WIN_LEN, 1)
    y     = data['y']
    print(f'    Computing RR features for {len(X_sig)} signals ...')
    X_feat = _extract_features(X_sig)
    print(f'  Done: Signal {X_sig.shape}, Features {X_feat.shape}')
    return X_sig, X_feat, y


print('=' * 70)
print('LOADING DATA + EXTRACTING FEATURES')
print('=' * 70)
X_train, X_train_feat, y_train = _load_split('train')
X_val,   X_val_feat,   y_val   = _load_split('val')
X_test,  X_test_feat,  y_test  = _load_split('test')
print(f'\nShapes — Train: {X_train.shape} + {X_train_feat.shape}')


unique_cls, cls_counts = np.unique(y_train, return_counts=True)
n_total   = len(y_train)
n_classes = len(CLASSES)

raw_weights = {
    int(idx): n_total / (n_classes * cnt)
    for idx, cnt in zip(unique_cls, cls_counts)
}
class_weight_dict = {i: raw_weights.get(i, 1.0) for i in range(n_classes)}

print('Class distribution in training set:')
for idx, count in zip(unique_cls, cls_counts):
    print(f'  {CLASSES[idx]:4s}: {count:6,}  ({count/n_total*100:.1f}%)'
          f'  weight={class_weight_dict[idx]:.4f}')


def residual_block(x, filters, kernel_size, downsample=True):
    shortcut = x
    if downsample:
        shortcut = layers.Conv1D(filters, 1, strides=2, padding='same')(shortcut)
    elif shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding='same')(shortcut)

    conv_stride = 2 if downsample else 1
    _reg = tf.keras.regularizers.l2(1e-4)
    y = layers.Conv1D(filters, kernel_size, strides=conv_stride, padding='same',
                      kernel_regularizer=_reg)(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation('relu')(y)
    y = layers.Conv1D(filters, kernel_size, padding='same',
                      kernel_regularizer=_reg)(y)
    y = layers.BatchNormalization()(y)
    y = layers.Add()([shortcut, y])
    y = layers.Activation('relu')(y)
    y = layers.SpatialDropout1D(0.3)(y)
    return y


def build_dual_model(input_len=1024, n_features=7, n_classes=4):
    sig_input = layers.Input(shape=(input_len, 1), name='signal_in')

    x = layers.Conv1D(32, 7, padding='same')(sig_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = residual_block(x, filters=64,  kernel_size=31, downsample=True)
    x = residual_block(x, filters=128, kernel_size=15, downsample=True)
    x = residual_block(x, filters=256, kernel_size=5,  downsample=True)

    attn = layers.Dense(1, activation='tanh')(x)
    attn = layers.Flatten()(attn)
    attn = layers.Activation('softmax')(attn)
    attn = layers.RepeatVector(256)(attn)
    attn = layers.Permute([2, 1])(attn)
    x    = layers.Multiply()([x, attn])

    path_a = layers.GlobalMaxPooling1D()(x)

    t = layers.Conv1D(64,  kernel_size=3, dilation_rate=4,  padding='same')(x)
    t = layers.BatchNormalization()(t)
    t = layers.Activation('relu')(t)
    t = layers.Conv1D(64,  kernel_size=3, dilation_rate=8,  padding='same')(t)
    t = layers.BatchNormalization()(t)
    t = layers.Activation('relu')(t)
    t = layers.Conv1D(128, kernel_size=3, dilation_rate=16, padding='same')(t)
    t = layers.BatchNormalization()(t)
    t = layers.Activation('relu')(t)
    t = layers.Conv1D(128, kernel_size=3, dilation_rate=32, padding='same')(t)
    t = layers.BatchNormalization()(t)
    t = layers.Activation('relu')(t)
    path_b = layers.GlobalAveragePooling1D()(t)

    sig_features = layers.Concatenate()([path_a, path_b])
    sig_features = layers.Dropout(0.4)(sig_features)

    feat_input = layers.Input(shape=(n_features,), name='feat_in')
    y = layers.BatchNormalization()(feat_input)
    y = layers.Dense(128, activation='relu')(y)
    y = layers.Dropout(0.3)(y)
    y = layers.Dense(64,  activation='relu')(y)
    y = layers.Dropout(0.2)(y)
    y = layers.Dense(32,  activation='relu')(y)

    combined = layers.Concatenate()([sig_features, y])
    z = layers.Dense(256, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(0.001))(combined)
    z = layers.BatchNormalization()(z)
    z = layers.Dropout(0.4)(z)
    z = layers.Dense(128, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(0.001))(z)
    z = layers.Dropout(0.4)(z)
    output = layers.Dense(n_classes, activation='softmax')(z)

    model = models.Model(inputs=[sig_input, feat_input], outputs=output)
    return model

In [ ]:
from sklearn.metrics import precision_recall_fscore_support


class MacroF1Callback(tf.keras.callbacks.Callback):
    """Compute val_macro_f1 at end of each epoch and inject into logs.

    Must appear BEFORE ModelCheckpoint / EarlyStopping in the callbacks
    list so those callbacks see the metric when they fire.
    """
    def __init__(self, val_inputs, y_val):
        super().__init__()
        self._val_inputs = val_inputs
        self._y_val      = y_val

    def on_epoch_end(self, epoch, logs=None):
        y_pred = np.argmax(
            self.model.predict(self._val_inputs, verbose=0), axis=1
        )
        score = f1_score(self._y_val, y_pred, average='macro')
        logs['val_macro_f1'] = float(score)
        print(f'  val_macro_f1: {score:.4f}')


def _run_one_training(seed, run_index):
    """Build, train, evaluate and log one independent model. Returns summary dict."""
    _run_ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir = os.path.join(RUNS_DIR, f'run_{run_index:02d}_seed{seed}_{_run_ts}')
    os.makedirs(run_dir, exist_ok=True)
    print(f'\n' + '=' * 70)
    print(f'RUN {run_index + 1}/{N_RUNS}  |  seed={seed}  |  {run_dir}')
    print('=' * 70)

    tf.random.set_seed(seed)
    np.random.seed(seed)

    model = build_dual_model(WIN_LEN, 7, len(CLASSES))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    if run_index == 0:
        model.summary()

    _best_ckpt = os.path.join(run_dir, 'best_model_filtered.keras')
    _callbacks = [
        MacroF1Callback([X_val, X_val_feat], y_val),
        tf.keras.callbacks.ModelCheckpoint(
            _best_ckpt, save_best_only=True,
            monitor='val_macro_f1', mode='max', verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_macro_f1', mode='max', factor=0.5, patience=5,
            min_delta=0.002, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_macro_f1', mode='max', patience=12, min_delta=0.002,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(
            os.path.join(run_dir, 'training_log.csv'), separator=',', append=False,
        ),
    ]

    history = model.fit(
        [X_train, X_train_feat], y_train,
        validation_data=([X_val, X_val_feat], y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=_callbacks,
        class_weight=class_weight_dict,
        verbose=1,
    )

    _final_run = os.path.join(run_dir, 'final_model_dual.keras')
    model.save(_final_run)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history.history['loss'],        label='train loss')
    axes[0].plot(history.history['val_loss'],     label='val loss')
    axes[0].set_title(f'Loss (seed={seed})'); axes[0].legend(); axes[0].set_xlabel('Epoch')
    axes[1].plot(history.history['accuracy'],     label='train acc')
    axes[1].plot(history.history['val_accuracy'], label='val acc')
    axes[1].set_title(f'Accuracy (seed={seed})'); axes[1].legend(); axes[1].set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'learning_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

    test_loss, test_acc = model.evaluate([X_test, X_test_feat], y_test, verbose=1)
    y_pred_proba  = model.predict([X_test, X_test_feat], verbose=1)
    y_pred        = np.argmax(y_pred_proba, axis=1)

    report_str      = classification_report(y_test, y_pred, target_names=CLASSES, digits=4)
    precision_macro = precision_score(y_test, y_pred, average='macro')
    recall_macro    = recall_score(y_test,    y_pred, average='macro')
    f1_macro        = f1_score(y_test,        y_pred, average='macro')
    precision_wt    = precision_score(y_test, y_pred, average='weighted')
    recall_wt       = recall_score(y_test,    y_pred, average='weighted')
    f1_wt           = f1_score(y_test,        y_pred, average='weighted')
    print(report_str)

    _stopped_epoch   = len(history.history['loss'])
    _best_val_loss   = min(history.history['val_loss'])
    _best_val_acc    = max(history.history['val_accuracy'])
    _best_val_f1     = max(history.history['val_macro_f1'])

    with open(os.path.join(run_dir, 'classification_report.txt'), 'w') as _f:
        _f.write(f'Run index  : {run_index}\n')
        _f.write(f'Seed       : {seed}\n')
        _f.write(f'Timestamp  : {_run_ts}\n')
        _f.write(f'Stopped ep : {_stopped_epoch}\n')
        _f.write(f'Best val loss     : {_best_val_loss:.4f}\n')
        _f.write(f'Best val acc      : {_best_val_acc:.4f}\n')
        _f.write(f'Best val macro-F1 : {_best_val_f1:.4f}\n')
        _f.write(f'Test loss  : {test_loss:.4f}\n')
        _f.write(f'Test acc   : {test_acc:.4f}\n\n')
        _f.write(report_str)
        _f.write(f'\nPrecision macro    : {precision_macro:.4f}\n')
        _f.write(f'Recall    macro    : {recall_macro:.4f}\n')
        _f.write(f'F1-score  macro    : {f1_macro:.4f}\n')
        _f.write(f'Precision weighted : {precision_wt:.4f}\n')
        _f.write(f'Recall    weighted : {recall_wt:.4f}\n')
        _f.write(f'F1-score  weighted : {f1_wt:.4f}\n')

    _p_pc, _r_pc, _f_pc, _s_pc = precision_recall_fscore_support(
        y_test, y_pred, labels=list(range(len(CLASSES))))
    _metrics_rows = []
    for _i, _cls in enumerate(CLASSES):
        _tp = int(np.sum((y_pred == _i) & (y_test == _i)))
        _fp = int(np.sum((y_pred == _i) & (y_test != _i)))
        _fn = int(np.sum((y_pred != _i) & (y_test == _i)))
        _tn = int(np.sum((y_pred != _i) & (y_test != _i)))
        _metrics_rows.append({
            'class': _cls,
            'TP': _tp, 'FP': _fp, 'FN': _fn, 'TN': _tn,
            'precision': round(float(_p_pc[_i]), 4),
            'recall':    round(float(_r_pc[_i]), 4),
            'f1':        round(float(_f_pc[_i]), 4),
            'support':   int(_s_pc[_i]),
        })
    with open(os.path.join(run_dir, 'per_class_metrics.csv'), 'w', newline='') as _f:
        _w = _csv.DictWriter(_f, fieldnames=list(_metrics_rows[0].keys()))
        _w.writeheader(); _w.writerows(_metrics_rows)

    cm      = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for _ax, _data, _fmt, _title in [
        (axes[0], cm,      'd',   f'Confusion Matrix — counts (seed={seed})'),
        (axes[1], cm_norm, '.2f', f'Confusion Matrix — row-normalised (seed={seed})'),
    ]:
        sns.heatmap(_data, annot=True, fmt=_fmt, cmap='Blues',
                    xticklabels=CLASSES, yticklabels=CLASSES, ax=_ax)
        _ax.set_xlabel('Predicted'); _ax.set_ylabel('Actual'); _ax.set_title(_title)
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
    plt.show()
    with open(os.path.join(run_dir, 'confusion_matrix.csv'), 'w', newline='') as _f:
        _w = _csv.writer(_f)
        _w.writerow(['actual \\ predicted'] + CLASSES)
        for _i, _row in enumerate(cm):
            _w.writerow([CLASSES[_i]] + list(_row))

    _meta = {
        'run_index':          run_index,
        'seed':               seed,
        'run_ts':             _run_ts,
        'run_dir':            run_dir,
        'stopped_epoch':      _stopped_epoch,
        'best_val_loss':      round(_best_val_loss,        4),
        'best_val_acc':       round(_best_val_acc,         4),
        'best_val_macro_f1':  round(_best_val_f1,          4),
        'test_loss':          round(float(test_loss),      4),
        'test_acc':           round(float(test_acc),       4),
        'precision_macro':    round(precision_macro,       4),
        'recall_macro':       round(recall_macro,          4),
        'f1_macro':           round(f1_macro,              4),
        'precision_weighted': round(precision_wt,          4),
        'recall_weighted':    round(recall_wt,             4),
        'f1_weighted':        round(f1_wt,                 4),
        'n_train':            int(len(y_train)),
        'n_val':              int(len(y_val)),
        'n_test':             int(len(y_test)),
        'batch_size':         BATCH_SIZE,
        'learning_rate':      LEARNING_RATE,
        'epochs_max':         EPOCHS,
    }
    with open(os.path.join(run_dir, 'run_meta.json'), 'w') as _f:
        _json.dump(_meta, _f, indent=2)

    print(f'  stopped_epoch={_stopped_epoch}  best_val_f1={_best_val_f1:.4f}'
          f'  test_acc={test_acc:.4f}  f1_macro={f1_macro:.4f}')
    print(f'  Artefacts saved to {run_dir}')
    tf.keras.backend.clear_session()
    return _meta


all_run_summaries = []
for _run_idx, _seed in enumerate(RUN_SEEDS):
    _summary = _run_one_training(_seed, _run_idx)
    all_run_summaries.append(_summary)

_summary_ts   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
_summary_path = os.path.join(RUNS_DIR, f'multi_run_summary_{_summary_ts}.csv')
os.makedirs(RUNS_DIR, exist_ok=True)
with open(_summary_path, 'w', newline='') as _f:
    _w = _csv.DictWriter(_f, fieldnames=list(all_run_summaries[0].keys()))
    _w.writeheader(); _w.writerows(all_run_summaries)

print('\n' + '=' * 70)
print(f'ALL {N_RUNS} RUNS COMPLETE')
print('=' * 70)
print(f'Summary CSV: {_summary_path}\n')
print(f'{"run":>4}  {"seed":>5}  {"stopped":>7}  {"val_acc":>8}  {"test_acc":>9}  {"f1_macro":>8}')
for _s in all_run_summaries:
    print(f'{_s["run_index"]:>4}  {_s["seed"]:>5}  {_s["stopped_epoch"]:>7}'
          f'  {_s["best_val_acc"]:>8.4f}  {_s["test_acc"]:>9.4f}  {_s["f1_macro"]:>8.4f}')

print(f'\n=== Block 7 complete ===')

---
## Block 8 — TFLite Conversion

Converts each run's `best_model_filtered.keras` to two TFLite formats:

| Variant | Optimisation |
|---------|-------------|
| Standard | Full float32 |
| Quantised | Dynamic-range quantisation (weights int8, activations float32) |

Output files saved to each run directory in `RUNS_DIR`:
```
run_XX_seedY_<ts>/
  best_model.tflite
  best_model_quant.tflite
```

In [ ]:
print('\n' + '=' * 70)
print('BLOCK 8 — TFLite Conversion')
print('=' * 70)

_tflite_rows = []

for _s in all_run_summaries:
    _run_dir   = _s['run_dir']
    _run_label = os.path.basename(_run_dir)
    _keras_path = os.path.join(_run_dir, 'best_model_filtered.keras')
    _std_path   = os.path.join(_run_dir, 'best_model.tflite')
    _quant_path = os.path.join(_run_dir, 'best_model_quant.tflite')

    if not os.path.exists(_keras_path):
        print(f'[SKIP] {_run_label}: best_model_filtered.keras not found')
        continue

    print(f'\n--- {_run_label} ---')

    print('  Loading Keras model ...')
    _model = tf.keras.models.load_model(_keras_path, compile=False)

    print('  Converting to standard TFLite ...')
    _conv = tf.lite.TFLiteConverter.from_keras_model(_model)
    _std_buf = _conv.convert()
    with open(_std_path, 'wb') as _f:
        _f.write(_std_buf)

    print('  Converting to quantised TFLite (dynamic-range) ...')
    _conv = tf.lite.TFLiteConverter.from_keras_model(_model)
    _conv.optimizations = [tf.lite.Optimize.DEFAULT]
    _quant_buf = _conv.convert()
    with open(_quant_path, 'wb') as _f:
        _f.write(_quant_buf)

    _kb = lambda p: os.path.getsize(p) / 1024
    _keras_kb = _kb(_keras_path)
    _std_kb   = _kb(_std_path)
    _quant_kb = _kb(_quant_path)

    print(f'  Keras  : {_keras_kb:>8.1f} KB')
    print(f'  TFLite : {_std_kb:>8.1f} KB  ({100*_std_kb/_keras_kb:.0f}% of Keras)')
    print(f'  Quant  : {_quant_kb:>8.1f} KB  ({100*_quant_kb/_keras_kb:.0f}% of Keras)')
    print(f'  Saved  : {_std_path}')
    print(f'           {_quant_path}')

    _tflite_rows.append({
        'run':        _run_label,
        'keras_kb':   round(_keras_kb, 1),
        'std_kb':     round(_std_kb,   1),
        'quant_kb':   round(_quant_kb, 1),
    })

    tf.keras.backend.clear_session()

# Summary table
if _tflite_rows:
    print('\n' + '=' * 70)
    print(f'{"run":<45}  {"keras KB":>8}  {"std KB":>7}  {"quant KB":>8}')
    print('-' * 70)
    for _row in _tflite_rows:
        print(f'{_row["run"]:<45}  {_row["keras_kb"]:>8.1f}  '
              f'{_row["std_kb"]:>7.1f}  {_row["quant_kb"]:>8.1f}')

print('\n=== Block 8 — TFLite conversion complete ===')